# Team Strategy & Orchestration Hub

Welcome to the project orchestration notebook. To prevent Git conflicts, ensure reproducibility, and avoid Jupyter Out-Of-Memory (OOM) crashes, **do not write long training loops directly in this notebook.** This notebook serves as our team's architectural blueprint and experiment execution tracker. It summarizes our shared data insights, establishes strict data preprocessing rules, and provides the CLI commands to trigger the isolated Python training scripts located in the `task_X/src/` directories.

## The Objective
* **Task A:** Binary Classification (Human vs. AI)
* **Task B:** Multi-Class Classification (30+ AI Generators)
* **Task C:** Advanced Classification (Human, AI, Adversarial, Hybrid)

---
## Shared Team Insights & Preprocessing Rules
Before implementing an approach, all team members must ensure their `src/dataset.py` or training scripts adhere to these findings from `EDA_and_Heuristics.ipynb`:

1. **Semantic Overlap:** PCA maps show massive vocabulary overlap between Human and AI code. Simple Bag-of-Words (TF-IDF) models will likely underperform. Deep contextual embeddings (e.g., CodeBERT) are highly recommended.
2. **Context Limits:** Human code frequently exceeds 512 tokens. Ensure your tokenizers are set to truncate appropriately (usually from the right).
3. **Class Imbalance:** Task B is heavily skewed. Consider using Focal Loss or Class Weights in your PyTorch training loops.

---
## TASK A

---
### Approach: Two-Stage Supervised Contrastive Learning (SupCon)

**Concept:** Rather than forcing a neural network to immediately guess "Human" or "AI", we first warp the mathematical space so that human code and AI code form distinct, isolated clusters. Once perfectly clustered, drawing a decision boundary between them becomes trivial.

**Strategy:** 
1. **Stage 1 (Representation Learning):** Feed code through UniXcoder and an MLP projection head. Train purely on SupCon Loss to form perfect 128-D clusters. 
2. **Stage 2 (Linear Classification):** Freeze the UniXcoder weights entirely and train a lightning-fast linear classifier on top of the 128-D embeddings to output final metrics.
* **Pros:** Mathematically bulletproof. No gradient interference between clustering and classifying. Highly robust against adversarial formatting (using the `--normalize` lexical ablation flag). Stage 2 iterates in seconds.
* **Cons:** Stage 1 is computationally heavy and requires a GPU. Moving to multi-class (Task B) will strictly require a custom Stratified Batch Sampler to prevent the "Batch Size Trap."

### Execution Commands

```bash
# Stage 1: Representation Learning (Clustering)
python task_A/src/train_stage1.py \
    --batch_size 8 \
    --accumulation_steps 4 \
    --normalize

# Stage 2: Linear Classifier (Evaluation)
python task_A/src/train_stage2.py \
    --stage1_weights <path/to/weights/stage1.pt> \
    --batch_size 32 \
    --normalize
```